# 컨텍스트 관리와 오프로딩

요약과 큰 도구 결과 오프로딩이 컨텍스트 크기를 다루는 위치를 구분합니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.

In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
cwd = Path.cwd().resolve()
LAB = next(
    parent / 'labs/part6/ch27-deepagents'
    for parent in [cwd, *cwd.parents]
    if (parent / 'labs/part6/ch27-deepagents/pyproject.toml').is_file()
)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

큰 결과를 반환하는 도구와 200토큰 한도의 파일시스템 미들웨어를 준비합니다.

In [3]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemMiddleware

@tool
def large_result() -> str:
    """Return a deliberately large local research result."""
    return '근거 문장입니다. ' * 3000

# 작은 한도를 사용해 큰 도구 결과의 파일 이동을 관찰합니다.
workspace = LAB / 'workspace-05'
workspace.mkdir(parents=True, exist_ok=True)
backend = FilesystemBackend(root_dir=workspace)
filesystem = FilesystemMiddleware(backend=backend, tool_token_limit_before_evict=200)
agent = create_deep_agent(model=init_chat_model('openai:gpt-5.4-mini'), tools=[large_result], backend=backend, middleware=[filesystem])

## 단계 2

도구를 호출해 ToolMessage에 남은 안내와 최종 답을 출력합니다.

In [4]:
# 큰 결과를 호출한 뒤 핵심만 답하도록 요청합니다.
result = agent.invoke({'messages':[{'role':'user','content':'large_result를 한 번 호출하고 첫 문장의 뜻만 답하세요.'}]})
tool_messages = [m for m in result['messages'] if type(m).__name__ == 'ToolMessage' and m.name == 'large_result']
print('도구 메시지 길이:', len(str(tool_messages[0].content)))
print('도구 메시지:', str(tool_messages[0].content)[:500])
assert tool_messages

도구 메시지 길이: 1653
도구 메시지: Tool result too large, the result of this tool call call_fUpAEusf2otgbyeSxR6QhIUk was saved in the filesystem at this path: /large_tool_results/call_fUpAEusf2otgbyeSxR6QhIUk

You can read the result from the filesystem by using the read_file tool, but make sure to only read part of the result at a time.

You can do this by specifying an offset and limit in the read_file tool call. For example, to read the first 100 lines, you can use the read_file tool with offset=0 and limit=100.

Here is a pre


## 단계 3

원문 대신 파일 경로가 남았는지와 실제 저장 파일을 확인합니다.

In [5]:
# 원문 대신 파일 경로가 남고 실제 파일이 생성됐는지 확인합니다.
import re

notice = str(tool_messages[0].content)
match = re.search(r'path: (/[^\s]+)', notice)
assert match is not None
saved_path = workspace / match.group(1).lstrip('/')
stored = saved_path.read_text(encoding='utf-8')
expected = '근거 문장입니다. ' * 3000
print('이번 생성 파일:', saved_path.relative_to(workspace).as_posix())
print('저장 본문 길이:', len(stored))
assert len(notice) < 3000
assert saved_path.is_file() and stored == expected
assert result['messages'][-1].text

이번 생성 파일: large_tool_results/call_fUpAEusf2otgbyeSxR6QhIUk
저장 본문 길이: 30000
